In [1]:
import sqlite3
import pandas as pd
import numpy as np

# 1. Establish connection to a local SQLite database file in the data folder
db_path = "../data/zaf_economic_data.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print(f"Database successfully connected/created at: {db_path}")

Database successfully connected/created at: ../data/zaf_economic_data.db


In [2]:
# Load the raw datasets
df_inf_raw = pd.read_csv("../data/inflation.csv")
df_pov_raw = pd.read_csv("../data/poverty.csv")

# Filter for South Africa (REF_AREA == 'ZAF')
df_inf_zaf = df_inf_raw[df_inf_raw['REF_AREA'] == 'ZAF'].copy()
df_pov_zaf = df_pov_raw[df_pov_raw['REF_AREA'] == 'ZAF'].copy()

# Identify year columns (all columns that are 4-digit numbers)
inf_year_cols = [c for c in df_inf_zaf.columns if c.isdigit()]
pov_year_cols = [c for c in df_pov_zaf.columns if c.isdigit()]

# Melt Inflation: wide format -> long format
df_inf_clean = df_inf_zaf.melt(
    id_vars=['REF_AREA', 'REF_AREA_LABEL', 'INDICATOR_LABEL'],
    value_vars=inf_year_cols,
    var_name='year',
    value_name='inflation_rate'
)
df_inf_clean['year'] = df_inf_clean['year'].astype(int)
df_inf_clean['inflation_rate'] = pd.to_numeric(df_inf_clean['inflation_rate'], errors='coerce')
df_inf_clean = df_inf_clean.dropna(subset=['inflation_rate']).reset_index(drop=True)

# Melt Poverty: wide format -> long format
df_pov_clean = df_pov_zaf.melt(
    id_vars=['REF_AREA', 'REF_AREA_LABEL', 'INDICATOR_LABEL'],
    value_vars=pov_year_cols,
    var_name='year',
    value_name='poverty_rate'
)
df_pov_clean['year'] = df_pov_clean['year'].astype(int)
df_pov_clean['poverty_rate'] = pd.to_numeric(df_pov_clean['poverty_rate'], errors='coerce')
df_pov_clean = df_pov_clean.dropna(subset=['poverty_rate']).reset_index(drop=True)

# Display clean preview table
display(df_inf_clean.head(4).round(2))

,REF_AREA,REF_AREA_LABEL,INDICATOR_LABEL,year,inflation_rate
0,ZAF,South Africa,"Inflation, average consumer prices, Percent ch...",1980,14.24
1,ZAF,South Africa,"Inflation, average consumer prices, Percent ch...",1981,15.35
2,ZAF,South Africa,"Inflation, average consumer prices, Percent ch...",1982,13.72
3,ZAF,South Africa,"Inflation, average consumer prices, Percent ch...",1983,12.80


In [3]:
# 1. Reset tables if they already exist to prevent duplicates on rerun
cursor.execute("DROP TABLE IF EXISTS inflation_data;")
cursor.execute("DROP TABLE IF EXISTS poverty_data;")

# 2. Create Inflation Table Schema
cursor.execute("""
CREATE TABLE inflation_data (
    country_code TEXT NOT NULL,
    country_name TEXT NOT NULL,
    indicator TEXT NOT NULL,
    year INTEGER NOT NULL,
    inflation_rate REAL NOT NULL
);
""")

# 3. Create Poverty Table Schema
cursor.execute("""
CREATE TABLE poverty_data (
    country_code TEXT NOT NULL,
    country_name TEXT NOT NULL,
    indicator TEXT NOT NULL,
    year INTEGER NOT NULL,
    poverty_rate REAL NOT NULL
);
""")
conn.commit()

# 4. Ingest data into the established schemas
df_inf_clean.rename(columns={
    'REF_AREA': 'country_code',
    'REF_AREA_LABEL': 'country_name',
    'INDICATOR_LABEL': 'indicator'
}).to_sql('inflation_data', conn, if_exists='append', index=False)

df_pov_clean.rename(columns={
    'REF_AREA': 'country_code',
    'REF_AREA_LABEL': 'country_name',
    'INDICATOR_LABEL': 'indicator'
}).to_sql('poverty_data', conn, if_exists='append', index=False)

# 5. Query the database to verify tables and record counts (Interactive Table)
verification_query = """
SELECT 
    'inflation_data' AS "Table Name", 
    COUNT(*) AS "Total Records" 
FROM inflation_data
UNION ALL
SELECT 
    'poverty_data' AS "Table Name", 
    COUNT(*) AS "Total Records" 
FROM poverty_data;
"""
df_status = pd.read_sql_query(verification_query, conn)
display(df_status)

,Table Name,Total Records
0,inflation_data,50
1,poverty_data,7


In [4]:
# Query 1: Join Inflation and Poverty on 'year'
join_query = """
SELECT 
    i.country_name AS "Country",
    i.year AS "Year",
    ROUND(i.inflation_rate, 2) AS "Inflation Rate (%)",
    ROUND(p.poverty_rate, 2) AS "Poverty Rate (%)"
FROM inflation_data i
INNER JOIN poverty_data p ON i.year = p.year
ORDER BY i.year ASC;
"""
df_combined = pd.read_sql_query(join_query, conn)

# Query 2: Summary statistics using SQL aggregations (AVG, MIN, MAX)
summary_query = """
SELECT 
    country_name AS "Country",
    COUNT(year) AS "Years Recorded",
    ROUND(AVG(inflation_rate), 2) AS "Mean Inflation (%)",
    ROUND(MIN(inflation_rate), 2) AS "Min Inflation (%)",
    ROUND(MAX(inflation_rate), 2) AS "Max Inflation (%)"
FROM inflation_data
GROUP BY country_name;
"""
df_summary = pd.read_sql_query(summary_query, conn)

# Display both tables in clean, styled interactive views
display(df_combined)
display(df_summary)

,Country,Year,Inflation Rate (%),Poverty Rate (%)
0,South Africa,1993,9.72,75.4
1,South Africa,2000,5.32,78.5
2,South Africa,2005,3.37,74.9
3,South Africa,2008,11.02,70.5
4,South Africa,2010,4.21,65.1
5,South Africa,2014,6.10,65.7
6,South Africa,2022,6.87,59.6


,Country,Years Recorded,Mean Inflation (%),Min Inflation (%),Max Inflation (%)
0,South Africa,50,8.11,1.39,18.1


In [5]:
# 1. Safe UPDATE: Updating a targeted record using a strict WHERE clause
update_query = """
UPDATE inflation_data
SET inflation_rate = 5.25
WHERE country_code = 'ZAF' AND year = 2020;
"""
cursor.execute(update_query)
conn.commit()

# Verify the update in SQLite
df_updated = pd.read_sql_query("""
SELECT 
    country_name AS "Country", 
    year AS "Year", 
    inflation_rate AS "Updated Inflation Rate (%)"
FROM inflation_data
WHERE country_code = 'ZAF' AND year = 2020;
""", conn)

# 2. Safe DELETE: Removing invalid/projected records with a safe WHERE boundary
delete_query = """
DELETE FROM inflation_data
WHERE year > 2030;
"""
cursor.execute(delete_query)
conn.commit()

# Verify the table boundaries post-deletion
df_delete_check = pd.read_sql_query("""
SELECT 
    MIN(year) AS "Min Year", 
    MAX(year) AS "Max Year (Post-Delete)", 
    COUNT(*) AS "Remaining Inflation Records"
FROM inflation_data;
""", conn)

# Display interactive verification tables
display(df_updated)
display(df_delete_check)

,Country,Year,Updated Inflation Rate (%)
0,South Africa,2020,5.25


,Min Year,Max Year (Post-Delete),Remaining Inflation Records
0,1980,2029,50


In [6]:
# Close the database connection cleanly
conn.close()
print("Database connection closed successfully.")

Database connection closed successfully.
